In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.optim as optim

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [4]:
train_datasets = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_datasets = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

c:\Users\oguzh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
train_datasets, val_datasets = random_split(train_datasets, lengths= [0.85,0.15])

len(train_datasets), len(val_datasets)

(42500, 7500)

In [6]:
train_dataloader = DataLoader(
    dataset= train_datasets,
    batch_size= 32,
    shuffle = True,
    num_workers= 4,
    pin_memory= True,
)

val_dataloader = DataLoader(
    dataset= val_datasets,
    batch_size= 32,
    shuffle= False,
    num_workers= 4,
    pin_memory= True
)

test_dataloader = DataLoader(
    dataset= test_datasets,
    batch_size= 32,
    shuffle = False,
    num_workers= 4,
    pin_memory= True
)

## MobilNetV1

In [7]:
class DeptWiseBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

  def forward(self, x):
      return self.block(x)

In [8]:
class MobileNetV1(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            DeptWiseBlock(32, 64, stride=1),

            DeptWiseBlock(64, 128, stride=2),
            DeptWiseBlock(128, 128, stride=1),

            DeptWiseBlock(128, 256, stride=2),
            DeptWiseBlock(256, 256, stride=1),

            DeptWiseBlock(256, 512, stride=2),

            DeptWiseBlock(512, 512, stride=1),
            DeptWiseBlock(512, 512, stride=1),
            DeptWiseBlock(512, 512, stride=1),
            DeptWiseBlock(512, 512, stride=1),
            DeptWiseBlock(512, 512, stride=1),

            DeptWiseBlock(512, 1024, stride=2),
            DeptWiseBlock(1024, 1024, stride=1),
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            1024,
            num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

num_classes = len(train_datasets.dataset.classes)
print(num_classes)
print(list(train_datasets.dataset.classes))

MobileNetV1_model = MobileNetV1(num_classes= num_classes).to(device= device)

10
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [9]:
optimizer = optim.Adam(MobileNetV1_model.parameters(), lr= 0.01)
criterion = nn.CrossEntropyLoss()

In [10]:
# Train && Validation Loop
epochs = 10

for epoch in range(epochs):
    MobileNetV1_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for (X, y) in train_dataloader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        
        y_pred = MobileNetV1_model(X)
        loss = criterion(y_pred, y)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        _, predicted = torch.max(y_pred, 1)
        train_total += y.size(0)
        train_correct += (predicted == y).sum().item()

    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    MobileNetV1_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for (X, y) in val_dataloader:
            X = X.to(device)
            y = y.to(device)

            y_pred = MobileNetV1_model(X)
            loss = criterion(y_pred, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(y_pred, 1)
            val_total += y.size(0)
            val_correct += (predicted == y).sum().item()

    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f} - Train Acc: {train_epoch_acc:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f} - Val Acc: {val_epoch_acc:.4f}")

Epoch [1/10] | Train Loss: 1.5895 - Train Acc: 0.4093 | Val Loss: 1.5359 - Val Acc: 0.4483
Epoch [2/10] | Train Loss: 1.0213 - Train Acc: 0.6379 | Val Loss: 0.9921 - Val Acc: 0.6559


KeyboardInterrupt: 